# APS Failure Prediction - Predictive Maintenance for Scania Trucks

## Overview

The Air Pressure System (APS) is a crucial component for Scania Trucks such that it generates pressurized air for braking and gear changes. The goal of this project was to predict APS failures before they happen thus letting fleet operators schedule maintenance proactively, therefore, avoiding costly roadside breakdowns. The task at hand is a cost senitive binary classification problem such that a false negative costs `$500` where a truck breaks down on the road and a false positive costs `$10` such that a mechanic performs an unnecessary inspection on a healthy truck.

The dataset is heavily imbalanced (59:1) with 170 anonymized sensor features. This makes for a realistic test of handling class imbalance and aligning model evaluation with business cost rather than raw accuracy.

## Approach

1. Median imputation for missing sensor readings
2. EDA of feature variability and correlation
3. Compared tree based methods including Random Forest, XGBoost, and XGBoost with SMOTE
4. Evaluated using domain specific cost metric

## Setup
1. Imports pipeline modules
2. Loads the project configuration

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_config, load_train_data, load_test_data
from src.preprocessing import encode_labels, split_features_target, impute_missing_values
from src.eda import compute_coefficient_of_variation, plot_correlation_matrix, get_top_cv_features, plot_scatterplot, plot_boxplot
from src.models import train_random_forest_classifier, train_xgb_classifier, train_SMOTE_xgb_classifier
from src.evaluate import compute_misclassification_rate, compute_cost, compute_cv_error, plot_roc_curve

config = load_config()

## Data Loading
1. Loads training and test sets
2. Encodes class labels to binary (pos -> 1, neg -> 0)
3. Split both training and test set each into features and targets

In [ ]:
train_df = load_train_data(config['data'])
test_df = load_test_data(config['data'])

train_df = encode_labels(train_df,
                         config['data']['target_column'],
                         config['data']['positive_label'],
                         config['data']['negative_label'])
X_train, y_train = split_features_target(train_df, config['data']['target_column'])

test_df = encode_labels(test_df,
                        config['data']['target_column'],
                        config['data']['positive_label'],
                        config['data']['negative_label'])
X_test, y_test = split_features_target(test_df, config['data']['target_column'])

## Preprocessing
1. Impute missing sensor values using the median
2. Fit on training data only to prevent leakage

In [ ]:
X_train_imputed, X_test_imputed = impute_missing_values(X_train,
                                                        X_test,
                                                        config['preprocessing']['imputation_strategy'])

## Exploratory Data Analysis
1. Computes coefficient of variation to see variability of feature
2. Obtains top coefficient of variation features
3. Creates correlation matrix
4. Plots scatterplot of features
5. Plots boxplot of features

In [ ]:
cv = compute_coefficient_of_variation(X_train_imputed)
top_features = get_top_cv_features(cv, 13)
plot_correlation_matrix(X_train_imputed)
plot_scatterplot(X_train_imputed, y_train, top_features)
plot_boxplot(X_train_imputed,y_train, top_features)

In [6]:
from src.models import train_random_forest_classifier, train_xgb_classifier, train_SMOTE_xgb_classifier

rf_model = train_random_forest_classifier(X_train_imputed, y_train, config['random_forest'])
print(f'Random Forest Trained, OOB score: {rf_model.oob_score_}')

Random Forest Trained, OOB score: 0.9938333333333333
